In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


<style>
  code {
    color: #0971c5;
  }
</style>

<h1 style="text-align:center; font-size:3em;font-weight:bold;"> Car Racing: Proximal Policy Optimization</h1>

<hr style="border: 1px solid #ccc; margin: 20px 0;">

## Description
Reference: https://gymnasium.farama.org/environments/box2d/car_racing/

<div style="text-align: center;">
    <img src="images/car_racing.gif" style="width: 400px;">
</div>

This is a pixel-based racing environment where an agent (a car) has to learn to navigate a randomly generated track each episode.

<table>
  <tr>
    <td>Action Space</td>
    <td><code>Box([-1. 0. 0.], 1.0, (3,), float32)</code></td>
  </tr>
  <tr>
    <td>Observation Space</td>
    <td><code>Box(0, 255, (96, 96, 3), uint8)</code></td>
  </tr>
</table>

## Action Space
**If continuous there are 3 actions:**
[steer,gas,brake]

| Index | Action Element | Range         | Description                                   |
| ----- | -------------- | ------------- | --------------------------------------------- |
| 0     | `steer`        | [-1.0, 1.0] | -1 = full left, 0 = straight, +1 = full right |
| 1     | `gas`          | [0.0, 1.0]  | 0 = no throttle, 1 = full throttle            |
| 2     | `brake`        | [0.0, 1.0] | 0 = no brake, 1 = full brake                  |


</br>

**If discrete there are 5 action:**
| Index | Action        |
|-------|---------------|
| 0     | Do nothing    |
| 1     | Steer left    |
| 2     | Steer right   |
| 3     | Gas           |
| 4     | Brake         |

## Observation Space

$96 \times 96$ RGB image that captures the car and track. 


## Arguments


- `lap_complete_percent=0.95` dictates the percentage of tiles that must be visited by the agent before a lap is considered complete.

- `domain_randomize=False` enables the domain randomized variant of the environment. In this scenario, the background and track colours are different on every reset.

- `continuous=True` converts the environment to use discrete or continuous action space.

## Rewards

The reward is $-0.1$ every frame and $+1000/N$ for every track tile visited.
- $N =$ Total number of tiles visited in the track

For example, if you finished in 732 frames, then the reward is: $1000-0.1\times 732 =926.8$ points

## Episode Termination

Episodes ends when:
- All the tiles have been visited.
- Car goes outside the playfield (far off the track) when it will receive $-100$ reward and die.

<hr style="border: 1px solid #ccc; margin: 20px 0;">


# 0. Imports and Setup

In [ ]:
# !pip install gymnasium
# !pip install gymnasium[classic-control]
# !pip install swig
# !pip install wheel setuptools pip --upgrade
# !pip install gymnasium[box2d]
# !pip install stable-baselines3
# !pip install stable-baselines3[extra] --upgrade
# !pip install tqdm rich
# !pip install "gymnasium[other]"
# !pip install moviepy

# IMPORTS

import gymnasium as gym
import matplotlib.pyplot as plt
import torch
import time

import imageio # to save video

import gymnasium
from gymnasium.wrappers import RecordVideo
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.atari_wrappers import WarpFrame
from stable_baselines3.common.vec_env import VecFrameStack, VecVideoRecorder
from stable_baselines3.common.callbacks import BaseCallback, EvalCallback
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.vec_env import VecTransposeImage
import os
import torch
import numpy
import platform
import stable_baselines3
import matplotlib
import matplotlib.pyplot
from platform import python_version
from importlib.metadata import version

import cv2
import numpy as np

from stable_baselines3.common.vec_env import VecNormalize
from stable_baselines3.common.env_util import make_vec_env

from utils.utils import * 


## Set up

In [ ]:
# Defining path names
env_str = "CarRacing-v3"
base_log_dir = "./model_logs"

| learning_rate    | n_steps | batch_size | n_epochs | gamma   | gae_lambda | clip_range |normalize_advantage|ent_coef|
|-------|----------------|---------|-------------|----------------|-------------|-------------|--|--|
|0.0003 | 2048      | 64    | 10        | 0.99     | 0.95     | 0.2      |True|0.0|

- learning_rate: it can also be a function
- n_steps: number of steps to run for each environment per update (i.e. rollout buffer size = n_steps *n_envs, where n_envs is number of environment copies running in parallel)
- batch_size: Minibatch size
- n_epochs: Number of epoch when optimizing the surrogate loss
- gamma: Discount factor
- clip_range: Clipping parameter, it can be a function of the current progress remaining (1-0)
- normalize_advantage: Yes or No
- ent_coef: Entropy coefficient for the loss calculation

In [ ]:
# Defining path names
env_str = "CarRacing-v3"
base_log_dir = "./model_logs"

In [ ]:
log_dir = get_unique_log_dir(base_log_dir, env_str)

print("Logging to:", log_dir)

env = create_env(env_id=env_str)
env_val = create_env(env_id=env_str, seed=100)

eval_callback = create_eval_callback(env=env_val, best_model_save_path=log_dir, log_path=log_dir)


In [ ]:

# Model Parameters
model = PPO('CnnPolicy',env)


# Train the model
model.learn(total_timesteps=500_000,
            progress_bar=True,
            callback=eval_callback)

# Save the model
model.save(os.path.join(log_dir, "ppo_car_racing"))


env.close()
env_val.close()


In [ ]:
# Load the evaluations.npz file
data = numpy.load(os.path.join(log_dir, "evaluations.npz"))

# Extract the relevant data
timesteps = data['timesteps']
results = data['results']

# Calculate the mean and standard deviation of the results
mean_results = numpy.mean(results, axis=1)
std_results = numpy.std(results, axis=1)

# Plot the results
matplotlib.pyplot.figure()
matplotlib.pyplot.plot(timesteps, mean_results)
matplotlib.pyplot.fill_between(timesteps,
                               mean_results - std_results,
                               mean_results + std_results,
                               alpha=0.3)

matplotlib.pyplot.xlabel('Timesteps')
matplotlib.pyplot.ylabel('Mean Reward')
matplotlib.pyplot.title(f"PPO Performance on {env_str}")
matplotlib.pyplot.show()

> What could be happening:
>- Zero entropy: 
>    Too deterministic, locked in on a suboptimal policy.
>
>- Overfitting to early behaviors: 
>    It found a decent path to 800, and kept repeating it.ut when the environment varied slightly, the policy wasn’t robust to handle it.
>
>- Catastrophic forgetting: </br>
>    PPO is on-policy, so bad updates after 800 might have overwritten good behaviors, especially given that entropy is zero.

## 1.1 Evaluate 30 Episodes

### 1.1.1 Deterministic

In [ ]:
# Create Evaluation CarRacing environment
env = create_env(env_str, n_envs=1, seed=0, wrapper_class=WarpFrame)


# Load the best model and evaluate it
best_model_path = os.path.join(log_dir, "best_model.zip")
best_model = PPO.load(best_model_path, env=env)

mean_reward, std_reward = evaluate_policy(best_model, env, n_eval_episodes=30)
print(f"Best Model - Mean reward: {mean_reward:.2f} +/- {std_reward:.2f}")

scores = []
for _ in range(30):
    returns, _ = evaluate_policy(best_model, env, n_eval_episodes=1, deterministic=True)
    scores.append(returns)


In [ ]:
# DETERMINISTIC POLICY
plt.hist(scores, bins=20)
plt.xlabel("Score")
plt.ylabel("Frequency")
plt.title("Distribution of Scores for the Model")
plt.show()

### 1.1.2 Stochastic

In [ ]:
# Create Evaluation CarRacing environment
env = create_env(env_str, n_envs=1, seed=0, wrapper_class=WarpFrame)


# Load the best model and evaluate it
best_model_path = os.path.join(log_dir, "best_model.zip")
best_model = PPO.load(best_model_path, env=env)

mean_reward, std_reward = evaluate_policy(best_model, env, n_eval_episodes=30, deterministic=False)
print(f"Best Model - Mean reward: {mean_reward:.2f} +/- {std_reward:.2f}")

scores = []
for _ in range(30):
    returns, _ = evaluate_policy(best_model, env, n_eval_episodes=1, deterministic=False)
    scores.append(returns)


In [ ]:
plt.hist(scores, bins=20)
plt.xlabel("Score")
plt.ylabel("Frequency")
plt.title("Distribution of Scores for the Model")
plt.show()

## 1.2 Save Video

In [ ]:
log_dir="./model_logs/CarRacing-v3_model_0"

In [ ]:
video_folder = "./videos/car_racing_ppo"

env = create_env(env_str, n_envs=1,seed=0, wrapper_class=WarpFrame)

env = VecVideoRecorder(
    env,
    video_folder,
    record_video_trigger=lambda step: step == 0,
    video_length=1000,
    name_prefix="ppo_model_1"
)

best_model = PPO.load(os.path.join(log_dir, "best_model.zip"), env=env)

obs = env.reset()
for _ in range(1000):
    action, _ = best_model.predict(obs, deterministic=True)
    obs, _, dones, _ = env.step(action)
    if dones[0]:
        break

env.close()
print(f"Video saved to: {video_folder}")

In [ ]:
latest_video = show_latest_video(video_folder=video_folder)
print("Played:", latest_video)

# 2. Add Entropy Coefficient

| learning_rate    | n_steps | batch_size | n_epochs | gamma   | gae_lambda | clip_range |normalize_advantage|ent_coef|
|-------|----------------|---------|-------------|----------------|-------------|-------------|--|--|
|0.0003 | 2048      | 64    | 10        | 0.99     | 0.95     | 0.2      |True|0.001|

In [ ]:
log_dir = get_unique_log_dir(base_log_dir, env_str)

print("Logging to:", log_dir)

env = create_env(env_id=env_str)
env_val = create_env(env_id=env_str, seed=100)

eval_callback = create_eval_callback(env=env_val, best_model_save_path=log_dir, log_path=log_dir)


In [ ]:
# Model Parameters
model = PPO('CnnPolicy',env, ent_coef= 0.001)


# Train the model
model.learn(total_timesteps=500_000,
            progress_bar=True,
            callback=eval_callback)

# Save the model
model.save(os.path.join(log_dir, "ppo_car_racing"))

#139m


In [ ]:
data = numpy.load(os.path.join(log_dir, "evaluations.npz"))

timesteps = data['timesteps']
results = data['results']

mean_results = numpy.mean(results, axis=1)
std_results = numpy.std(results, axis=1)

matplotlib.pyplot.figure()
matplotlib.pyplot.plot(timesteps, mean_results)
matplotlib.pyplot.fill_between(timesteps,
                               mean_results - std_results,
                               mean_results + std_results,
                               alpha=0.3)

matplotlib.pyplot.xlabel('Timesteps')
matplotlib.pyplot.ylabel('Mean Reward')
matplotlib.pyplot.title(f"PPO Performance on {env_str}")
matplotlib.pyplot.show()

> Training was quite unstable, but it seemed to be improving. Maybe with more timesteps it could get better and get stable. But for now we will lower the entropy.

## 2.1 Evaluate 30 times

### 2.1.1 Deterministic

In [ ]:
env = create_env(env_str, n_envs=1, seed=0, wrapper_class=WarpFrame)


best_model_path = os.path.join(log_dir, "best_model.zip")
best_model = PPO.load(best_model_path, env=env)

mean_reward, std_reward = evaluate_policy(best_model, env, n_eval_episodes=30)
print(f"Best Model - Mean reward: {mean_reward:.2f} +/- {std_reward:.2f}")

scores = []
for _ in range(30):
    returns, _ = evaluate_policy(best_model, env, n_eval_episodes=1, deterministic=True)
    scores.append(returns)


In [ ]:
# DETERMINISTIC POLICY
plt.hist(scores, bins=20)
plt.xlabel("Score")
plt.ylabel("Frequency")
plt.title("Distribution of Scores for the Model")
plt.show()

### 2.1.2 Stochastic 

In [ ]:
lod_dir="./model_logs/CarRacing-v3_model_1"

In [ ]:
env = create_env(env_str, n_envs=1, seed=0, wrapper_class=WarpFrame)


best_model_path = os.path.join(log_dir, "best_model.zip")
best_model = PPO.load(best_model_path, env=env)

mean_reward, std_reward = evaluate_policy(best_model, env, n_eval_episodes=30, deterministic=False)
print(f"Best Model - Mean reward: {mean_reward:.2f} +/- {std_reward:.2f}")

scores = []
for _ in range(30):
    returns, _ = evaluate_policy(best_model, env, n_eval_episodes=1, deterministic=False)
    scores.append(returns)

plt.hist(scores, bins=20)
plt.xlabel("Score")
plt.ylabel("Frequency")
plt.title("Distribution of Scores for the Model")
plt.show()

## 2.2 Save Video

In [ ]:
log_dir="./model_logs/CarRacing-v3_model_1"

In [ ]:
video_folder = "./videos/car_racing_ppo"

env = create_env(env_str, n_envs=1,seed=0, wrapper_class=WarpFrame)

env = VecVideoRecorder(
    env,
    video_folder,
    record_video_trigger=lambda step: step == 0,
    video_length=1000,
    name_prefix="ppo_model_2"
)

best_model = PPO.load(os.path.join(log_dir, "best_model.zip"), env=env)

obs = env.reset()
for _ in range(1000):
    action, _ = best_model.predict(obs, deterministic=True)
    obs, _, dones, _ = env.step(action)
    if dones[0]:
        break

env.close()
print(f"Video saved to: {video_folder}")

In [ ]:
latest_video = show_latest_video(video_folder=video_folder)
print("Played:", latest_video)

# 3. Entropy = 0.0001

In [ ]:
log_dir = get_unique_log_dir(base_log_dir, env_str)

print("Logging to:", log_dir)

env = create_env(env_id=env_str)
env_val = create_env(env_id=env_str, seed=100)

eval_callback = create_eval_callback(env=env_val, best_model_save_path=log_dir, log_path=log_dir)


In [ ]:
# Model Parameters
model = PPO('CnnPolicy',env, ent_coef=0.0005)


# Train the model
model.learn(total_timesteps=500_000,
            progress_bar=True,
            callback=eval_callback)

# Save the model
model.save(os.path.join(log_dir, "ppo_car_racing"))

env.close()
env_val.close()


In [ ]:
data = numpy.load(os.path.join(log_dir, "evaluations.npz"))

timesteps = data['timesteps']
results = data['results']

mean_results = numpy.mean(results, axis=1)
std_results = numpy.std(results, axis=1)

matplotlib.pyplot.figure()
matplotlib.pyplot.plot(timesteps, mean_results)
matplotlib.pyplot.fill_between(timesteps,
                               mean_results - std_results,
                               mean_results + std_results,
                               alpha=0.3)

matplotlib.pyplot.xlabel('Timesteps')
matplotlib.pyplot.ylabel('Mean Reward')
matplotlib.pyplot.title(f"PPO Performance on {env_str}")
matplotlib.pyplot.show()

## 3.1 Evaluate 30 Episodes

### 3.1.1 Deterministic

In [ ]:
log_dir="./model_logs/CarRacing-v3_model_3"

In [ ]:
env = create_env(env_str, n_envs=1, seed=0, wrapper_class=WarpFrame)


best_model_path = os.path.join(log_dir, "best_model.zip")
best_model = PPO.load(best_model_path, env=env)

mean_reward, std_reward = evaluate_policy(best_model, env, n_eval_episodes=30, deterministic=True)
print(f"Best Model - Mean reward: {mean_reward:.2f} +/- {std_reward:.2f}")

scores = []
for _ in range(30):
    returns, _ = evaluate_policy(best_model, env, n_eval_episodes=1, deterministic=True)
    scores.append(returns)


In [ ]:
# DETERMINISTIC POLICY
plt.hist(scores, bins=20)
plt.xlabel("Score")
plt.ylabel("Frequency")
plt.title("Distribution of Scores for the Model")
plt.show()

### 3.1.2 Stochastic

In [ ]:
env = create_env(env_str, n_envs=1, seed=0, wrapper_class=WarpFrame)


best_model_path = os.path.join(log_dir, "best_model.zip")
best_model = PPO.load(best_model_path, env=env)

mean_reward, std_reward = evaluate_policy(best_model, env, n_eval_episodes=30, deterministic=False)
print(f"Best Model - Mean reward: {mean_reward:.2f} +/- {std_reward:.2f}")

scores = []
for _ in range(30):
    returns, _ = evaluate_policy(best_model, env, n_eval_episodes=1, deterministic=False)
    scores.append(returns)


In [ ]:
# Plot
plt.hist(scores, bins=20)
plt.xlabel("Score")
plt.ylabel("Frequency")
plt.title("Distribution of Scores for the Model")
plt.show()

## 3.2 Save Video

In [ ]:
log_dir="./model_logs/CarRacing-v3_model_3"

In [ ]:
video_folder = "./videos/car_racing_ppo"

env = create_env(env_str, n_envs=1,seed=0, wrapper_class=WarpFrame)

env = VecVideoRecorder(
    env,
    video_folder,
    record_video_trigger=lambda step: step == 0,
    video_length=1000,
    name_prefix="ppo_model_3"
)

best_model = PPO.load(os.path.join(log_dir, "best_model.zip"), env=env)

obs = env.reset()
for _ in range(1000):
    action, _ = best_model.predict(obs, deterministic=True)
    obs, _, dones, _ = env.step(action)
    if dones[0]: 
        break

env.close()
print(f"Video saved to: {video_folder}")

In [ ]:
latest_video = show_latest_video(video_folder=video_folder)
print("Played:", latest_video)

# 4. Higher rollout

In [ ]:
log_dir = get_unique_log_dir(base_log_dir, env_str)

print("Logging to:", log_dir)

env = create_env(env_id=env_str)
env_val = create_env(env_id=env_str, seed=100)

eval_callback = create_eval_callback(env=env_val, best_model_save_path=log_dir, log_path=log_dir)


In [ ]:
# Model Parameters
model = PPO('CnnPolicy',env, ent_coef=0.0005, n_steps=4096, batch_size=128)


# Train the model
model.learn(total_timesteps=500_000,
            progress_bar=True,
            callback=eval_callback)

# Save the model
model.save(os.path.join(log_dir, "ppo_car_racing"))

env.close()
env_val.close()

# 162m


> This is not getting to a really high score, it could be it needs more time to train to be able to get there.

In [ ]:
data = numpy.load(os.path.join(log_dir, "evaluations.npz"))

timesteps = data['timesteps']
results = data['results']

mean_results = numpy.mean(results, axis=1)
std_results = numpy.std(results, axis=1)

matplotlib.pyplot.figure()
matplotlib.pyplot.plot(timesteps, mean_results)
matplotlib.pyplot.fill_between(timesteps,
                               mean_results - std_results,
                               mean_results + std_results,
                               alpha=0.3)

matplotlib.pyplot.xlabel('Timesteps')
matplotlib.pyplot.ylabel('Mean Reward')
matplotlib.pyplot.title(f"PPO Performance on {env_str}")
matplotlib.pyplot.show()

## 4.1 Evalute 30 Episodes

### 4.1.1 Deterministic

In [ ]:
env = create_env(env_str, n_envs=1, seed=0, wrapper_class=WarpFrame)


best_model_path = os.path.join(log_dir, "best_model.zip")
best_model = PPO.load(best_model_path, env=env)

mean_reward, std_reward = evaluate_policy(best_model, env, n_eval_episodes=30)
print(f"Best Model - Mean reward: {mean_reward:.2f} +/- {std_reward:.2f}")

scores = []
for _ in range(30):
    returns, _ = evaluate_policy(best_model, env, n_eval_episodes=1, deterministic=True)
    scores.append(returns)

plt.hist(scores, bins=20)
plt.xlabel("Score")
plt.ylabel("Frequency")
plt.title("Distribution of Scores for the Model")
plt.show()

### 4.1.2 Stochastic

In [ ]:
env = create_env(env_str, n_envs=1, seed=0, wrapper_class=WarpFrame)


best_model_path = os.path.join(log_dir, "best_model.zip")
best_model = PPO.load(best_model_path, env=env)

mean_reward, std_reward = evaluate_policy(best_model, env, n_eval_episodes=30, deterministic=False)
print(f"Best Model - Mean reward: {mean_reward:.2f} +/- {std_reward:.2f}")

scores = []
for _ in range(30):
    returns, _ = evaluate_policy(best_model, env, n_eval_episodes=1, deterministic=False)
    scores.append(returns)

plt.hist(scores, bins=20)
plt.xlabel("Score")
plt.ylabel("Frequency")
plt.title("Distribution of Scores for the Model")
plt.show()

## 4.2 Save Video

In [ ]:
log_dir="./model_logs/CarRacing-v3_model_4"

In [ ]:
video_folder = "./videos/car_racing_ppo"

env = create_env(env_str, n_envs=1,seed=0, wrapper_class=WarpFrame)

env = VecVideoRecorder(
    env,
    video_folder,
    record_video_trigger=lambda step: step == 0,
    video_length=1000,
    name_prefix="ppo_model_4"
)

best_model = PPO.load(os.path.join(log_dir, "best_model.zip"), env=env)

obs = env.reset()
for _ in range(1000):
    action, _ = best_model.predict(obs, deterministic=True)
    obs, _, dones, _ = env.step(action)
    if dones[0]: 
        break

env.close()
print(f"Video saved to: {video_folder}")

In [ ]:
latest_video = show_latest_video(video_folder=video_folder)
print("Played:", latest_video)